# PPOCRv4 fine-tuned rec → ONNX
使用 trainingOcr repo 的工具，直接把 model.pdparams 轉成 RapidOCR 可用的 .onnx

In [ ]:
# 1. 安裝環境（CPU 版 paddle，轉換不需要 GPU）
!pip install paddlepaddle==3.0.0 -q
!python -c "import paddle; print('paddle OK:', paddle.__version__)"
!pip install "paddle2onnx>=1.3.0" onnxruntime -q
!python -c "import paddle2onnx; print('paddle2onnx OK')"

In [ ]:
# 2. Clone trainingOcr repo（含 export 工具與 config）
!git clone https://github.com/install88/trainingOcr.git /content/trainingOcr
%cd /content/trainingOcr
# 安裝 PaddleOCR（export_model.py 需要它）
!pip install paddleocr -q
!ls configs/rec/

In [ ]:
# 3. 上傳 model.pdparams
from google.colab import files
uploaded = files.upload()   # 選 model.pdparams

import shutil, os
shutil.move('model.pdparams', '/content/trainingOcr/model.pdparams')
print('上傳完成:', os.path.getsize('/content/trainingOcr/model.pdparams') / 1e6, 'MB')

In [ ]:
# 4. 用官方 PaddleOCR export 工具
!pip install lmdb -q

import os, re

# 把 config 裡的 hardcode Windows 路徑換成 Colab 路徑
config_path = '/content/trainingOcr/configs/rec/PP-OCRv4_mobile_rec_finetune.yml'
with open(config_path, 'r') as f:
    cfg = f.read()

cfg = re.sub(
    r'character_dict_path:.*',
    'character_dict_path: /content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt',
    cfg
)

with open(config_path, 'w') as f:
    f.write(cfg)

print('config 路徑已修正：')
!grep character_dict_path {config_path}

# Export
!python /content/PaddleOCR/tools/export_model.py \
    -c {config_path} \
    -o Global.pretrained_model=/content/trainingOcr/model.pdparams \
       Global.save_inference_dir=/content/inference_rec

print('\n--- inference_rec contents ---')
!ls -lh /content/inference_rec/

In [ ]:
# 5. 轉 ONNX（PaddlePaddle 3.x 用 inference.json 而非 inference.pdmodel）
!paddle2onnx \
    --model_dir /content/inference_rec \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file /content/rec_finetuned.onnx \
    --opset_version 11

size_mb = os.path.getsize('/content/rec_finetuned.onnx') / 1024 / 1024
print(f'ONNX size: {size_mb:.1f} MB')

In [ ]:
# 6. 驗證 input shape
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('/content/rec_finetuned.onnx')
inp = sess.get_inputs()[0]
print(f'Input : {inp.name}  shape={inp.shape}  dtype={inp.type}')
print(f'Outputs: {[o.name for o in sess.get_outputs()]}')

dummy = np.zeros([1, 3, 48, 320], dtype=np.float32)
out = sess.run(None, {inp.name: dummy})
print(f'Output shape: {out[0].shape}')
print('ONNX OK')

In [ ]:
# 7. 下載
files.download('/content/rec_finetuned.onnx')
# 放到：C:/Users/insta/Desktop/date-recognition/models/rec_finetuned.onnx